# AGI [progress-measuring] EF (Executive Functions) Tasks

**Kaggle Community Benchmark · `tuple[int, int]` Count Task**

Evaluates Lean 4 proof generation against **7 assertions** derived from
the DeepMind *Measuring Progress Toward AGI: A Cognitive Framework*
(Burnell et al., 2026) §7.8 Executive Functions sub-abilities.

In [1]:
## AGI_EF_Tasks__Basic -- v1
## agi-ef-tasks--basic.ipynb
## https://www.kaggle.com/code/selnour/agi-ef-tasks
## https://www.kaggle.com/code/selnour/agi-ef-x-task
## Return type: tuple[int, int] (passed_assertions, total_assertions)
## Ref: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
## Ref: DeepMind AGI Cognitive Framework (Burnell et al., 2026) Section 7.8

# DeepMind Cognitive Taxonomy §7.8 — Executive Functions

> *Higher-order cognitive abilities that enable goal-directed behavior
> by regulating and orchestrating thoughts and actions* (Diamond, 2013).

## Seven Assertions ↔ §7.8.x Sub-Abilities

| # | Ref | Sub-Ability | Lean 4 Operationalization |
|---|-----|-------------|---------------------------|
| 1 | 7.8.1 | **Goal Setting & Maintenance** | Proof preserves theorem/lemma goal statement |
| 2 | 7.8.2 | **Planning** | Multi-step tactic sequence (>=2 distinct tactics) |
| 3 | 7.8.3 | **Inhibitory Control** | Suppresses `sorry`/`admit` shortcuts |
| 4 | 7.8.4 | **Cognitive Flexibility** | Uses tactics from >=2 distinct categories |
| 5 | 7.8.5 | **Conflict Resolution** | Manages competing subgoals (case splits, focused goals) |
| 6 | 7.8.6 | **Working Memory** | Tracks intermediate state (`have`/`let`/`obtain`/`suffices`) |
| 7 | 7.8   | **Composite EF Integration** | Overall structural validity (proof body + balanced syntax) |

### References

- Diamond, A. (2013). Executive functions. Annual Review of Psychology, 64, 135-168.
- Owen, A. M. (1997). Cognitive planning in humans.
- Buschman, T. J. & Miller, E. K. (2014). Goal-direction and top-down control.
- Bari, A. & Robbins, T. W. (2013). Inhibition and impulsivity.
- Braem, S. & Egner, T. (2018). Getting a grip on cognitive flexibility.
- Botvinick, M. M. et al. (2001). Conflict monitoring and cognitive control.
- Miyake, A. et al. (2000). The unity and diversity of executive functions.
- Mattar, M. G. & Lengyel, M. (2022). Planning in the brain.
- Burnell, R. et al. (2026). Measuring Progress Toward AGI. Google DeepMind.

### Dataset: vSPACE v_train_extended.csv

- **Total proof tasks**: 1,239
- **Difficulty categories**: 6 (basic, advanced, coq_based, basic_core, autonomous, augmented)
- **Domain**: Formal verification of cryptographic protocols, election systems, ZKPs
- **Format**: Lean 4 proof tasks with `-- !benchmark @start/@end` markers

In [2]:
!pip install -q wget matplotlib seaborn

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any  # built-in tuple used for return type
import json
import re

# Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [4]:
CSV_PATH = Path('/kaggle/input/datasets/selnour/v-train-extended/v_train_extended.csv')
df = pd.read_csv(CSV_PATH)

# Filter to only 'basic' difficulty for benchmarking
df = df[df['difficulty'] == 'basic'].reset_index(drop=True)

print(f"Loaded {len(df)} proof tasks (basic difficulty only)")
print(f"\nDifficulty distribution:")
print(df['difficulty'].value_counts())
print(f"\nColumns: {list(df.columns)}")

Loaded 108 proof tasks (basic difficulty only)

Difficulty distribution:
difficulty
basic    108
Name: count, dtype: int64

Columns: ['id', 'description', 'lean_code', 'signature', 'metadata', 'tests', 'reject_inputs', 'difficulty']


## Proof Extraction & 7.8.x Assertion Functions

The LLM response is typically wrapped in markdown code fences.
We extract the raw Lean 4 code before evaluating against the 7 assertions.

In [5]:
# =====================================================================
# PROOF EXTRACTION -- strip markdown fencing from LLM response
# =====================================================================

def extract_lean_proof(response):
    """Extract Lean 4 code from LLM response, stripping markdown fences."""
    if not response or not isinstance(response, str):
        return ""
    # Try ```lean4 or ```lean blocks
    matches = re.findall(r'```(?:lean4?)\s*\n(.*?)```', response, re.DOTALL)
    if matches:
        return matches[-1].strip()
    # Fallback: any ``` block
    matches = re.findall(r'```\s*\n(.*?)```', response, re.DOTALL)
    if matches:
        return matches[-1].strip()
    # Last resort: return raw (might already be clean)
    return response.strip()


# =====================================================================
# TACTIC ANALYSIS UTILITIES
# =====================================================================

# Lean 4 tactics organized by cognitive category
TACTIC_CATEGORIES = {
    'introduction':   ['intro', 'intros', 'rintro'],
    'application':    ['apply', 'exact', 'refine', 'use'],
    'rewriting':      ['rw', 'rewrite', 'rfl', 'subst'],
    'simplification': ['simp', 'norm_num', 'ring', 'omega', 'decide', 'trivial'],
    'destruction':    ['cases', 'induction', 'rcases', 'obtain', 'match'],
    'construction':   ['constructor', 'left', 'right', 'split'],
    'hypothesis':     ['have', 'let', 'suffices', 'show', 'calc'],
    'goal_mgmt':      ['focus', 'next', 'all_goals', 'any_goals', 'try', 'first'],
}

ALL_TACTICS = []
for cat_tactics in TACTIC_CATEGORIES.values():
    ALL_TACTICS.extend(cat_tactics)


def find_tactics_used(proof):
    """Return list of distinct tactics found in proof."""
    found = []
    lower = proof.lower()
    for tactic in ALL_TACTICS:
        if re.search(r'\b' + re.escape(tactic.lower()) + r'\b', lower):
            found.append(tactic)
    return found


def find_categories_used(proof):
    """Return set of tactic categories used in proof."""
    cats = set()
    lower = proof.lower()
    for cat_name, cat_tactics in TACTIC_CATEGORIES.items():
        for t in cat_tactics:
            if re.search(r'\b' + re.escape(t.lower()) + r'\b', lower):
                cats.add(cat_name)
                break
    return cats


# =====================================================================
# 7.8.x ASSERTION FUNCTIONS -- each returns bool
# =====================================================================

def assert_goal_setting(proof):
    """7.8.1 Goal Setting & Maintenance (Buschman & Miller, 2014).
    The LLM must preserve the theorem/lemma goal statement."""
    goal_kw = ['theorem', 'lemma', 'def', 'example', 'instance']
    return any(kw in proof.lower() for kw in goal_kw)


def assert_planning(proof):
    """7.8.2 Planning (Owen, 1997; Mattar & Lengyel, 2022).
    Multi-step tactic sequencing -- requires >= 2 distinct tactics."""
    return len(find_tactics_used(proof)) >= 2


def assert_inhibitory_control(proof):
    """7.8.3 Inhibitory Control (Bari & Robbins, 2013).
    Suppresses sorry/admit shortcuts in the proof body."""
    lower = proof.lower()
    # Find start of proof body
    body_start = -1
    for marker in [':= by', ':=by', ' by\n', ' by ', ':= ', '\nby\n', '\nby ']:
        idx = lower.find(marker)
        if idx >= 0:
            body_start = idx
            break
    if body_start < 0:
        return False  # no proof body = inhibitory control not demonstrated
    body = lower[body_start:]
    shortcuts = ['sorry', 'admit']
    return not any(s in body for s in shortcuts)


def assert_cognitive_flexibility(proof):
    """7.8.4 Cognitive Flexibility (Braem & Egner, 2018).
    Uses tactics from >= 2 distinct categories."""
    return len(find_categories_used(proof)) >= 2


def assert_conflict_resolution(proof):
    """7.8.5 Conflict Resolution (Botvinick et al., 2001).
    Manages competing subgoals via case splits, focused goals, or intermediates."""
    markers = ['case ', 'next', 'all_goals', 'focus', 'have ', 'let ']
    lower = proof.lower()
    # Also check for Lean 4 focus dot
    if ' \u00b7 ' in proof or '\n  \u00b7 ' in proof or '\n\u00b7 ' in proof:
        return True
    return any(m in lower for m in markers)


def assert_working_memory(proof):
    """7.8.6 Working Memory (Diamond, 2013; Baddeley, 2000).
    Tracks intermediate state via have/let/obtain/suffices/show."""
    wm_markers = ['have ', 'let ', 'obtain ', 'suffices ', 'show ', 'calc', 'this']
    lower = proof.lower()
    if any(m in lower for m in wm_markers):
        return True
    # Hypothesis references like h1, h_prime
    if re.search(r'\bh[_\d]', lower):
        return True
    return False


def assert_structural_validity(proof):
    """7.8 Composite EF Integration.
    Overall structural validity: proof body + balanced syntax."""
    if len(proof.strip()) < 10:
        return False
    has_body = 'by' in proof or ':=' in proof
    balanced = proof.count('{') == proof.count('}')
    balanced_parens = proof.count('(') == proof.count(')')
    return has_body and balanced and balanced_parens


# Ordered list of all 7 assertion functions
EF_ASSERTIONS = [
    ("7.8.1 Goal Setting & Maintenance",  assert_goal_setting),
    ("7.8.2 Planning",                     assert_planning),
    ("7.8.3 Inhibitory Control",           assert_inhibitory_control),
    ("7.8.4 Cognitive Flexibility",        assert_cognitive_flexibility),
    ("7.8.5 Conflict Resolution",          assert_conflict_resolution),
    ("7.8.6 Working Memory",               assert_working_memory),
    ("7.8   Composite EF Integration",     assert_structural_validity),
]

NUM_ASSERTIONS = len(EF_ASSERTIONS)  # 7

print(f"Defined {NUM_ASSERTIONS} EF assertion functions:")
for name, fn in EF_ASSERTIONS:
    print(f"  {name}: {fn.__name__}")

Defined 7 EF assertion functions:
  7.8.1 Goal Setting & Maintenance: assert_goal_setting
  7.8.2 Planning: assert_planning
  7.8.3 Inhibitory Control: assert_inhibitory_control
  7.8.4 Cognitive Flexibility: assert_cognitive_flexibility
  7.8.5 Conflict Resolution: assert_conflict_resolution
  7.8.6 Working Memory: assert_working_memory
  7.8   Composite EF Integration: assert_structural_validity


## AGI_EF_Tasks -- `tuple[int, int]` Count Task

This task iterates over the full dataset (1,239 proof tasks), prompts the
LLM once per row, extracts the Lean 4 proof from the response, and
evaluates 7 EF assertions per proof.

**Return**: `(passed_assertions, total_assertions)` where
`total_assertions = num_rows x 7`.

The leaderboard displays this as a ratio (e.g., `4231/8673 = 0.488`).

In [6]:
# ## v1.1 ## no progress loggiing
# @kbench.task(name="AGI_EF_Tasks", store_task=True)
# def AGI_EF_Tasks(llm) -> tuple[int, int]:
#     """
#     EF-DA-TASK: Planning -- Multi-Step Proof Construction.

#     Returns:
#         tuple[int, int]: (passed_assertions, total_assertions)
#     """
#     # """
#     # EF-DA-TASK: Planning -- Multi-Step Proof Construction.

#     # Measures 7 Executive Function sub-abilities (DeepMind 7.8.1-7.8.6 + composite)
#     # across all 1,239 vSPACE Lean 4 proof tasks.

#     # Returns:
#     #     tuple[int, int]: (passed_assertions, total_assertions)
#     # """
#     total_passed = 0
#     total_possible = 0

#     for idx, row in df.iterrows():
#         task_id    = str(row['id'])
#         desc       = str(row['description'])
#         lean_code  = str(row['lean_code'])
#         difficulty = str(row['difficulty'])

#         # -- Prompt the LLM -------------------------------------------------
#         prompt = f"""Complete the following Lean 4 proof.

# {desc}

# ```lean4
# {lean_code}
# ```

# Replace every `sorry` with a complete formal proof.
# Use appropriate tactics for difficulty level: {difficulty}.

# IMPORTANT: Return ONLY the complete Lean 4 code inside a single
# ```lean4 code block. No explanations, no commentary."""

#         try:
#             response = llm.prompt(prompt, max_tokens=2048, temperature=0.2)
#         except TypeError:
#             try:
#                 response = llm.prompt(prompt)
#             except Exception:
#                 response = ""
#         except Exception:
#             response = ""

#         if not isinstance(response, str):
#             response = str(response) if response is not None else ""

#         # -- Extract proof from markdown-wrapped response --------------------
#         proof = extract_lean_proof(response)

#         # -- Evaluate 7 EF assertions ---------------------------------------
#         for _name, assertion_fn in EF_ASSERTIONS:
#             total_possible += 1
#             try:
#                 if assertion_fn(proof):
#                     total_passed += 1
#             except Exception:
#                 pass  # assertion fails on error

#     if idx % 100 == 0:
#     print(f"  Row {idx}/{len(df)} — passed so far: {total_passed}/{total_possible}")
    
#     # -- Aggregate kbench assertion for leaderboard signal -------------------
#     kbench.assertions.assert_true(
#         total_possible > 0,
#         expectation="Benchmark must evaluate at least one proof task."
#     )

#     return (total_passed, total_possible)

In [7]:
## v1.2 ## progress console loggiing
import time

@kbench.task(name="AGI_EF_Tasks__Basic", store_task=True)
def AGI_EF_Tasks(llm) -> tuple[int, int]:
    """
    EF-DA-TASK: Planning -- Multi-Step Proof Construction.

    Returns:
        tuple[int, int]: (passed_assertions, total_assertions)
    """
    # """
    # Measures 7 Executive Function sub-abilities (DeepMind 7.8.1-7.8.6 + composite)
    # across all 1,239 vSPACE Lean 4 proof tasks.
    # """
    total_passed = 0
    total_possible = 0
    t_start = time.time()

    # Per-assertion running counters (7 slots)
    ef_passed = [0] * NUM_ASSERTIONS
    ef_total  = [0] * NUM_ASSERTIONS

    # Per-difficulty running counters
    diff_passed = {}
    diff_total  = {}

    # Track rows where ALL 7 assertions pass (perfect proofs)
    perfect_count = 0

    for idx, row in df.iterrows():
        task_id    = str(row['id'])
        desc       = str(row['description'])
        lean_code  = str(row['lean_code'])
        difficulty = str(row['difficulty'])

        # -- Prompt the LLM -------------------------------------------------
        prompt = f"""Complete the following Lean 4 proof.

{desc}

```lean4
{lean_code}
```

Replace every `sorry` with a complete formal proof.
Use appropriate tactics for difficulty level: {difficulty}.

IMPORTANT: Return ONLY the complete Lean 4 code inside a single
```lean4 code block. No explanations, no commentary."""

        try:
            response = llm.prompt(prompt, max_tokens=2048, temperature=0.2)
        except TypeError:
            try:
                response = llm.prompt(prompt)
            except Exception:
                response = ""
        except Exception:
            response = ""

        if not isinstance(response, str):
            response = str(response) if response is not None else ""

        # -- Extract proof from markdown-wrapped response --------------------
        proof = extract_lean_proof(response)

        # -- Evaluate 7 EF assertions ---------------------------------------
        row_passed = 0
        for a_idx, (_name, assertion_fn) in enumerate(EF_ASSERTIONS):
            total_possible += 1
            ef_total[a_idx] += 1
            diff_total.setdefault(difficulty, 0)
            diff_total[difficulty] = diff_total.get(difficulty, 0) + 1
            try:
                if assertion_fn(proof):
                    total_passed += 1
                    ef_passed[a_idx] += 1
                    diff_passed[difficulty] = diff_passed.get(difficulty, 0) + 1
                    row_passed += 1
            except Exception:
                pass

        if row_passed == NUM_ASSERTIONS:
            perfect_count += 1

        # -- Progress milestone every 100 rows -------------------------------
        if (idx + 1) % 100 == 0 or idx == len(df) - 1:
            elapsed = time.time() - t_start
            rows_done = idx + 1
            rate = rows_done / elapsed if elapsed > 0 else 0
            eta = (len(df) - rows_done) / rate if rate > 0 else 0

            pct = total_passed / total_possible * 100 if total_possible > 0 else 0

            print(f"\n{'='*60}")
            print(f"  MILESTONE: Row {rows_done}/{len(df)}  "
                  f"|  {elapsed:.0f}s elapsed  |  {rate:.1f} rows/s  "
                  f"|  ETA {eta:.0f}s")
            print(f"  Aggregate: {total_passed}/{total_possible} "
                  f"({pct:.1f}%)  |  Perfect 7/7: {perfect_count}/{rows_done}")
            print(f"  {'─'*56}")

            # Per-assertion breakdown (compact single-line each)
            for a_idx, (name, _fn) in enumerate(EF_ASSERTIONS):
                p = ef_passed[a_idx]
                t = ef_total[a_idx]
                apct = p / t * 100 if t > 0 else 0
                bar = '█' * int(apct // 10) + '░' * (10 - int(apct // 10))
                print(f"  {bar} {apct:5.1f}%  {name}")

            # Per-difficulty one-liner
            diff_summary = []
            for d in sorted(diff_total.keys()):
                dp = diff_passed.get(d, 0)
                dt = diff_total[d]
                dpct = dp / dt * 100 if dt > 0 else 0
                diff_summary.append(f"{d}={dpct:.0f}%")
            print(f"  {'─'*56}")
            print(f"  By difficulty: {' | '.join(diff_summary)}")
            print(f"{'='*60}")

    # -- Final kbench assertion for leaderboard signal -----------------------
    kbench.assertions.assert_true(
        total_possible > 0,
        expectation="Benchmark must evaluate at least one proof task."
    )

    return (total_passed, total_possible)


## Diagnostic: Single-Row Dry Run

Before submitting the full benchmark, test extraction + assertions
on one row to verify the pipeline works.

In [8]:
def dry_run_single_row(llm, row_idx=0):
    """Test the pipeline on a single row and print per-assertion results."""
    row = df.iloc[row_idx]

    prompt = f"""Complete the following Lean 4 proof.

{row['description']}

```lean4
{row['lean_code']}
```

Replace every `sorry` with a complete formal proof.
Use appropriate tactics for difficulty level: {row['difficulty']}.

IMPORTANT: Return ONLY the complete Lean 4 code inside a single
```lean4 code block. No explanations, no commentary."""

    try:
        response = llm.prompt(prompt, max_tokens=2048, temperature=0.2)
    except TypeError:
        response = llm.prompt(prompt)

    if not isinstance(response, str):
        response = str(response) if response is not None else ""

    proof = extract_lean_proof(response)

    print(f"Task:       {row['id']}")
    print(f"Difficulty: {row['difficulty']}")
    print(f"Response length:  {len(response)} chars")
    print(f"Extracted proof:  {len(proof)} chars")
    print(f"\n--- Extracted Proof (first 500 chars) ---")
    print(proof[:500])
    print(f"\n--- Tactics found ---")
    print(find_tactics_used(proof))
    print(f"\n--- Categories used ---")
    print(find_categories_used(proof))
    print(f"\n--- 7.8.x Assertion Results ---")

    passed = 0
    for name, fn in EF_ASSERTIONS:
        result = fn(proof)
        status = "PASS" if result else "FAIL"
        passed += int(result)
        print(f"  [{status}]  {name}")

    print(f"\nScore: {passed}/{NUM_ASSERTIONS}")
    return passed, NUM_ASSERTIONS


# Run diagnostic on row 0
dry_run_single_row(kbench.llm, row_idx=0)

Task:       verina_basic_70
Difficulty: basic
Response length:  10091 chars
Extracted proof:  10078 chars

--- Extracted Proof (first 500 chars) ---
-- !benchmark @start import type=solution

-- !benchmark @end import

-- !benchmark @start solution_aux

-- !benchmark @end solution_aux

-- !benchmark @start precond_aux

-- !benchmark @end precond_aux
@[reducible, simp]
def LinearSearch3_precond (a : Array Int) (P : Int -> Bool) : Prop :=
  -- !benchmark @start precond
  ∃ i, i < a.size ∧ P (a[i]!)
  -- !benchmark @end precond


-- !benchmark @start code_aux

-- !benchmark @end code_aux


def LinearSearch3 (a : Array Int) (P : Int -> Bool) (h_

--- Tactics found ---
['intro', 'apply', 'exact', 'refine', 'use', 'rewrite', 'rfl', 'subst', 'simp', 'cases', 'induction', 'rcases', 'split', 'have', 'let', 'show', 'first']

--- Categories used ---
{'destruction', 'goal_mgmt', 'construction', 'application', 'hypothesis', 'introduction', 'simplification', 'rewriting'}

--- 7.8.x Assertion Results

(5, 7)

## Full Benchmark Execution

Run the task on all 1,239 rows. The leaderboard will display the
`tuple[int, int]` result as a `Numerical_Result` ratio.

In [9]:
result = AGI_EF_Tasks.run(kbench.llm)

print(f"Status:        {result.status}")
print(f"Passed:        {result.passed}")
print(f"Error message: {result.error_message}")
print(f"Result type:   {type(result.result).__name__}")

if result.result is not None:
    passed, total = result.result
    print(f"\nAssertions passed: {passed}/{total}")
    if total > 0:
        print(f"Score:             {passed/total:.4f}")
else:
    print("\nNo result returned -- check error_message above.")


  MILESTONE: Row 100/108  |  1143s elapsed  |  0.1 rows/s  |  ETA 91s
  Aggregate: 663/700 (94.7%)  |  Perfect 7/7: 72/100
  ────────────────────────────────────────────────────────
  ██████████ 100.0%  7.8.1 Goal Setting & Maintenance
  █████████░  98.0%  7.8.2 Planning
  ████████░░  83.0%  7.8.3 Inhibitory Control
  █████████░  98.0%  7.8.4 Cognitive Flexibility
  █████████░  97.0%  7.8.5 Conflict Resolution
  ██████████ 100.0%  7.8.6 Working Memory
  ████████░░  87.0%  7.8   Composite EF Integration
  ────────────────────────────────────────────────────────
  By difficulty: basic=95%



  MILESTONE: Row 108/108  |  1242s elapsed  |  0.1 rows/s  |  ETA 0s
  Aggregate: 715/756 (94.6%)  |  Perfect 7/7: 78/108
  ────────────────────────────────────────────────────────
  ██████████ 100.0%  7.8.1 Goal Setting & Maintenance
  █████████░  97.2%  7.8.2 Planning
  ████████░░  84.3%  7.8.3 Inhibitory Control
  █████████░  97.2%  7.8.4 Cognitive Flexibility
  █████████░  96.3%  7.8.5 Conflict Resolution
  ██████████ 100.0%  7.8.6 Working Memory
  ████████░░  87.0%  7.8   Composite EF Integration
  ────────────────────────────────────────────────────────
  By difficulty: basic=95%


Status:        success
Passed:        True
Error message: None
Result type:   tuple

Assertions passed: 715/756
Score:             0.9458


## Summary

### Architecture

```
tuple[int, int] = (passed_assertions, total_assertions)
                = (sum of passed across 1239 rows x 7 assertions,
                   1239 x 7 = 8673)
```

### 7.8 Executive Functions <-> Assertions Mapping

| Assertion | Ref | What It Measures | Lean 4 Signal |
|-----------|-----|------------------|---------------|
| 1 | 7.8.1 Goal Setting & Maintenance | Preserves theorem goal | `theorem`/`lemma` keyword present |
| 2 | 7.8.2 Planning | Multi-step tactic sequence | >=2 distinct tactics used |
| 3 | 7.8.3 Inhibitory Control | Suppresses shortcuts | No `sorry`/`admit` in proof body |
| 4 | 7.8.4 Cognitive Flexibility | Diverse tactic types | >=2 tactic categories |
| 5 | 7.8.5 Conflict Resolution | Subgoal management | `case`/have/next present |
| 6 | 7.8.6 Working Memory | Intermediate state tracking | `have`/`let`/`obtain`/`suffices` |
| 7 | 7.8   Composite EF Integration | Structural validity | Proof body + balanced syntax |

### Key Fixes from Previous Versions

1. **Proof extraction** -- strips markdown fences before assertion evaluation
2. **Return type** -- `tuple[int, int]` (Count Task) instead of `dict` (unsupported on leaderboard)
3. **7 assertions** -- grounded in DeepMind 7.8 EF taxonomy, not ad-hoc checks
4. **Full dataset** -- iterates all 1,239 rows, not just `df.iloc[0]`
5. **Improved prompt** -- requests raw Lean 4 code only, no commentary

### References

- Burnell, R. et al. (2026). Measuring Progress Toward AGI: A Cognitive Framework. Google DeepMind.
- Diamond, A. (2013). Executive functions. Annual Review of Psychology, 64, 135-168.
- Kaggle Benchmarks SDK: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
- vSPACE Dataset: https://github.com/vSpaceVote/vSPACE